In [1]:
import sys
sys.path.insert(0,'..')
sys.path.insert(0,'../..')
import torch

from dig.sslgraph.utils import Encoder
from dig.sslgraph.evaluation import GraphUnsupervised
from dig.sslgraph.evaluation import Finetune
from dig.threedgraph.dataset import MoleculeNet
from dig.threedgraph.method import SchNet
from dig.sslgraph.method import GraphCL

import matplotlib.pyplot as plt

from rdkit import RDLogger 
RDLogger.DisableLog('rdApp.*')

from IPython.core.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))

import pandas as pd
from rdkit import Chem
import rdkit.Chem.AllChem as AllChem

import argparse

paser = argparse.ArgumentParser()
args = paser.parse_args("")

C:\Users\aiden\AppData\Local\Temp\ipykernel_21888\3539213752.py:18: DeprecationWarning: Importing display from IPython.core.display is deprecated since IPython 7.14, please import from IPython.display
  from IPython.core.display import display, HTML


In [2]:
### Finetune or rand init
args.finetune = False
args.seed = 2222

# File Path
args.model_path = './models/encoder-schnet_pretrain-esol_batch-400_proj-spherenet_cutoff-5.0_layers-2_filter-128_gau-50_z_dim-512_lr-0.001_\
aug_1-maskN_aug_2-maskN_aug_ratio-0.2_tau-0.2_optim-ExponentialLR_weight_decay-0_expo_gamma-0.95_dropout-0.0/enc_epoch-300_loss-5.699.pkl'

# Device
from dig.sslgraph.utils.device import pick_torch_device
args.device = pick_torch_device()
print("args.device:", args.device)

# Dataset
args.dataset = 'esol'

args.batch_size = 32

# Model
args.encoder = 'schnet'
args.cutoff = 5.0  # [5.0, 10.0]
args.num_layers = 2 # [2, 4]
args.num_filters = 128
args.num_gaussians = 50
args.z_dim = 64

args.edge_weight = False

# Learning
args.n_times = 2
args.n_folds = 3
args.f_epoch = 100
args.f_lr = 1e-3
args.aug_1, args.aug_2 = 'MMFFrandom', 'MMFFrandom'
args.aug_ratio = 0.2
args.tau = 0.2
args.proj = 'schnet'

# Regularization
args.dropout_rate = 0.0
#aug

args.f_optim = 'ExponentialLR' #['StepLR', ExponentialLR, 'Cosine']
args.f_weight_decay = 5e-5

#'StepLR'
args.f_lr_decay_step_size = 20  # 15 epoch 마다 lr * p_lr_decay_factor
args.f_lr_decay_factor = 0.5

# ExponentialLR 
args.expo_gamma = 0.95

# Cosine
args.T_0 = 100        # 최초 주기값
args.T_mult = 1      # 최초 주기값에 비해 얼만큼 주기를 늘려갈 것인지
args.eta_max = 0.05  # lr 최대값
args.T_up = 10        # Warm up 시 필요한 epoch 수(일반적으로 짧은 수)
args.gamma = 0.5     # 주기가 반복될수록 곱해지는 scale 값

args.batch_lst = [32]
args.cutoff_lst = [5.0]
args.num_layers_lst = [2]
args.num_filters_lst = [128]
args.num_gaussians_lst = [50]
args.z_dim_lst = [64]
args.dropout_rate_lst = [0.3]
args.target_lst = ['y']
args.f_lr_lst = [1e-3]
args.f_weight_decay_lst = [1e-3]

evaluator = Finetune(args=args, log_interval=10)
auc_m_lst, auc_sd_lst, paras, total, args = evaluator.grid_search(args)
#loss, sd = evaluator.evaluate()

args.device: xpu:0


c:\DGCL\3DGCL\examples\sslgraph\../..\dig\sslgraph\evaluation\finetune.py:256: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. The given 'InMemoryDataset' only references a subset of examples of the full dataset, but 'data' will contain information of the full dataset. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  self.dataset.data.y = self.dataset.data[self.target]


907 112 109


Finetune: epoch 83:  82%|████████▏ | 82/100 [01:53<00:24,  1.38s/it, FOLD=1.0, best_test_loss=0.891, best_val_loss=0.754, test_rmse=0.914, train_rmse=0.628, val_rmse=0.765]

Early stop at Epoch: 83 with final val loss: 0.7794


...

Finetune: epoch 96:  95%|█████████▌| 95/100 [02:50<00:08,  1.80s/it, FOLD=2.0, best_test_loss=0.887, best_val_loss=0.862, test_rmse=0.908, train_rmse=0.639, val_rmse=0.871]

Early stop at Epoch: 96 with final val loss: 0.8713


902 113 113


Finetune: epoch 79:  78%|███████▊  | 78/100 [02:20<00:39,  1.80s/it, FOLD=3.0, best_test_loss=0.785, best_val_loss=0.963, test_rmse=0.798, train_rmse=0.628, val_rmse=0.967]

Early stop at Epoch: 79 with final val loss: 0.9755
Total Results:  [((32, 5.0, 2, 64, 0.3), 0.905675743226749)]


In [3]:
# grid_search 결과를 파일로 저장 (분자별 실제값/예측값/오차)
import numpy as np
import pandas as pd
from pathlib import Path

required = ["total", "args"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "먼저 학습 셀을 실행하세요: `evaluator.grid_search(args)` 결과가 필요합니다. "
        f"누락 변수: {missing}"
    )

if not hasattr(args, "list_test_trues") or not hasattr(args, "list_test_preds"):
    raise RuntimeError(
        "현재 커널의 args에 예측 결과가 없습니다. 학습 셀을 먼저 실행한 뒤 다시 시도하세요."
    )

losses_grid = np.array([loss for (_, loss) in total])
best_i = int(np.argmin(losses_grid))

def _flatten_any(lst):
    parts = lst[best_i]
    out = []
    for a in parts:
        out.append(np.asarray(a).reshape(-1))
    return np.concatenate(out, axis=0)

y_true = _flatten_any(args.list_test_trues).astype(float)
y_pred = _flatten_any(args.list_test_preds).astype(float)
smiles = _flatten_any(args.list_test_smiles)

n = min(len(y_true), len(y_pred), len(smiles))
df = pd.DataFrame(
    {
        "smiles": smiles[:n],
        "y_true": y_true[:n],
        "y_pred": y_pred[:n],
    }
)
df["abs_error"] = (df["y_pred"] - df["y_true"]).abs()

out_dir = Path("./results") / args.dataset / args.encoder
out_dir.mkdir(parents=True, exist_ok=True)
out_csv = out_dir / "predictions_best_grid.csv"
df.to_csv(out_csv, index=False)

print(f"saved: {out_csv.resolve()}")
print(df.head(10))

saved: C:\DGCL\3DGCL\examples\sslgraph\results\esol\schnet\predictions_best_grid.csv
                                              smiles  y_true    y_pred  \
0                         c1ccc2cc3cc4ccccc4cc3cc2c1  -8.600 -6.990385   
1                 CC34CCC1C(=CCc2cc(O)ccc12)C3CCC4=O  -5.282 -3.862103   
2          Cc1ccc(OP(=O)(Oc2cccc(C)c2)Oc3ccccc3C)cc1  -6.010 -5.901109   
3  CC1OC(CC(O)C1O)OC2C(O)CC(OC2C)OC8C(O)CC(OC7CCC...  -4.081 -4.330021   
4  CC1OC(CC(O)C1O)OC2C(O)CC(OC2C)OC8C(O)CC(OC7CCC...  -5.293 -4.733954   
5  CC1CC2C3CC(F)C4=CC(=O)C=CC4(C)C3(F)C(O)CC2(C)C...  -5.613 -3.411434   
6  CC4CC3C2CCC1=CC(=O)C=CC1(C)C2(F)C(O)CC3(C)C4(O...  -4.900 -3.728487   
7  CC(=O)OCC(=O)C3(O)CCC4C2CCC1=CC(=O)C=CC1(C)C2C...  -4.370 -3.551843   
8  CCCCC(=O)OC3(C(C)CC4C2CCC1=CC(=O)C=CC1(C)C2(F)...  -4.710 -4.267294   
9  CC34CC(O)C1(F)C(CCC2=CC(=O)C=CC12C)C3CC(O)C4(O...  -3.680 -2.783487   

   abs_error  
0   1.609615  
1   1.419898  
2   0.108891  
3   0.249021  
4   0.559046  
5   2.2015